# AIM · 3주차 자율 과제
## Attention ① 관계를 계산하는 방법

**대상:** Python·PyTorch 기초 이수자 / **권장:** Google Colab Python 3, GPU 선택 가능.

행렬 곱, softmax, cross entropy를 복습한다. 외부 텍스트 다운로드 없이 교육용으로 직접 만든 문장 코퍼스를 사용한다. 결과는 언어 모델 원리 관찰용이며 자연어 능력 벤치마크가 아니다.

### 학습 목표
- Q·K·V와 scaled dot-product attention을 손으로 계산한다.
- causal mask와 텐서 shape를 검증한다.
- 단일 헤드 문자 언어 모델을 학습하고 생성한다.

### 실행 방법
1. Colab에서 `파일 → 노트북 업로드`로 이 파일을 엽니다.
2. GPU를 사용할 경우 `런타임 → 런타임 유형 변경`에서 GPU를 선택합니다.
3. 위에서 아래로 셀을 실행합니다. 이전 주차 파일이나 별도 Python 모듈은 필요 없습니다.
4. Colab 기본 torch·torchvision을 우선 사용합니다. import가 실패하면 새 런타임에서 시작하세요.
   로컬 환경은 루트 `환경설정.md`를 참고하세요. 데이터 최초 다운로드에는 인터넷이 필요합니다.
5. 학습이 길면 `TRAIN_N`, `EPOCHS` 또는 `STEPS`를 줄입니다. 작은 실험의 품질은 보장되지 않습니다.

**시간 운영:** 실습 셀 번호가 아니라 아래 § 구간으로 진행합니다. 준비·다운로드는 수업 시작에,
핵심 실행은 50–75분에, 관찰 답변은 75–85분에 수행합니다. 심화와 자율 과제는 수업 밖에서 진행합니다.

**재현성:** 고정 seed도 다른 GPU·라이브러리 버전의 완전히 같은 수치를 보장하지 않습니다.
실제 출력과 예측을 구분해 기록하세요. `outputs/`는 현재 실행 폴더에 생성됩니다.

### 자율 과제 안내

**A·B 기본 / C 비교 실험 / D 도전**입니다. 모두 선택이며 전부 하면 약 80–130분과 학습 시간이 필요합니다.
한 문제만 골라도 괜찮습니다. 이 노트북은 실습 파일 실행 없이 독립적으로 시작할 수 있습니다.

준비 코드는 제공하고, 아래 TODO 함수에 직접 구현합니다. 미완성 상태의 `return None`은 실행을 중단하지 않고 안내만 출력합니다.
코드를 작성하면 바로 다음 검사 셀을 실행하세요. 검사는 최소 계약만 확인하며 좋은 실험 해석을 자동 채점하지 않습니다.

**제출을 선택한다면:** 실행 결과가 남은 ipynb, 비교 표·그림, 아래 해석 5문장을 저장하세요.
test로 과제 조건을 반복 선택하지 말고 validation을 사용하세요.

## §1 · 환경 확인

첫 실행에서 버전과 device를 확인합니다. CPU도 지원하며 설치가 필요한 경우 환경설정 문서를 먼저 읽습니다.

In [1]:
import os, sys, math, random, copy, time, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import datasets, transforms
from IPython.display import display

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# QUICK=True: 수업용 일부 데이터. False: 데이터/학습 예산 확대.
QUICK = True
# 검증 제작자용 축소 모드. 수강생은 설정할 필요가 없습니다.
SMOKE = os.environ.get('AIM_SMOKE', '0') == '1'
DATA_ROOT = Path(os.environ.get('AIM_DATA_ROOT', './data'))
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
torch.set_num_threads(min(4, os.cpu_count() or 1))

def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_all()
print('Python', sys.version.split()[0], '| torch', torch.__version__,
      '| torchvision', torchvision.__version__, '| device', DEVICE)
print('QUICK:', QUICK, '| SMOKE:', SMOKE, '| data:', DATA_ROOT.resolve())

def image_grid(images, titles=None, cols=8, title='', normalized=False):
    images = images.detach().cpu()
    if normalized: images = (images * .5 + .5).clamp(0, 1)
    n = len(images); rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*1.35, rows*1.55), squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis('off')
        if i < n:
            im = images[i]
            ax.imshow(im.squeeze(0) if im.shape[0]==1 else im.permute(1,2,0),
                      cmap='gray', vmin=0, vmax=1)
            if titles is not None: ax.set_title(str(titles[i]), fontsize=8)
    fig.suptitle(title); fig.tight_layout(); plt.show()
    return fig

def num_params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

def cpu_state(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

Python 3.13.9 | torch 2.14.0+cpu | torchvision 0.29.0+cpu | device cpu
QUICK: True | SMOKE: False | data: D:\AIM\.study-data


In [2]:
BATCH = 8 if SMOKE else 32
CONTEXT = 16 if SMOKE else 32
STEPS = 6 if SMOKE else (240 if QUICK else 1500)

## §2 · 코퍼스, vocabulary, next-token target

수업용으로 새로 작성한 조합 문장을 사용합니다. 문장 단위로 먼저 분할하고 split 내부에서만 window를 만듭니다. exact 문장은 분리되지만 문법과 어휘는 공유됩니다. 실제 자연어 능력을 평가하는 자료는 아닙니다.

In [3]:
# 외부 저작물을 쓰지 않은 교육용 조합 문장. exact 문장 중복 없이 먼저 분할.
subjects=['the cat','the dog','a student','the teacher','a robot','my friend','the artist','a scientist']
verbs=['reads','finds','draws','builds','likes','studies']
objects=['a book','a small map','the new model','a red house','a quiet room','the blue box']
places=['in the lab','after class','near the river','every morning']
sentences=[f'{s} {v} {o} {p}.\n' for s in subjects for v in verbs for o in objects for p in places]
rng=random.Random(SEED); rng.shuffle(sentences)
n=len(sentences); a=int(.8*n); b=int(.9*n)
parts={'train':sentences[:a],'val':sentences[a:b],'test':sentences[b:]}
assert not (set(parts['train']) & set(parts['val']))
assert not (set(parts['train']) & set(parts['test']))
assert not (set(parts['val']) & set(parts['test']))
itos=['<UNK>']+sorted(set(''.join(parts['train'])))
stoi={c:i for i,c in enumerate(itos)}; VOCAB=len(itos)
def encode(text): return [stoi.get(c,0) for c in text]
def decode(ids): return ''.join(itos[int(i)] if int(i)!=0 else '�' for i in ids)
tokens={k:torch.tensor(encode(''.join(v)),dtype=torch.long) for k,v in parts.items()}
print('sentences:',{k:len(v) for k,v in parts.items()},'| vocab:',VOCAB)
print('tokens:',{k:len(v) for k,v in tokens.items()})
print('UNK fractions:',{k:(v==0).float().mean().item() for k,v in tokens.items()})
print('train example:',parts['train'][0])

def lm_batch(split='train',batch=BATCH,context=CONTEXT):
    data=tokens[split]; starts=torch.randint(len(data)-context,(batch,))
    x=torch.stack([data[i:i+context] for i in starts])
    y=torch.stack([data[i+1:i+context+1] for i in starts])
    return x.to(DEVICE),y.to(DEVICE)

x_example,y_example=lm_batch(batch=1)
print('input :',repr(decode(x_example[0])))
print('target:',repr(decode(y_example[0])))
assert torch.equal(x_example[:,1:],y_example[:,:-1])

sentences: {'train': 921, 'val': 115, 'test': 116} | vocab: 28
tokens: {'train': 38852, 'val': 4852, 'test': 4920}
UNK fractions: {'train': 0.0, 'val': 0.0, 'test': 0.0}
train example: the dog reads a red house after class.

input : ' after class.\na robot draws a sm'
target: 'after class.\na robot draws a sma'


## §3 · attention 직접 구현

`scaled_attention`은 True인 위치를 차단하는 `blocked`를 사용합니다. softmax는 key 축에 적용합니다.

In [4]:
def scaled_attention(q,k,v,causal=True):
    scores=q @ k.transpose(-2,-1) / math.sqrt(q.size(-1))
    if causal:
        blocked=torch.ones(q.size(-2),k.size(-2),device=q.device,dtype=torch.bool).triu(1)
        scores=scores.masked_fill(blocked,float('-inf'))
    weights=scores.softmax(dim=-1)
    return weights @ v,weights

class BigramLM(nn.Module):
    def __init__(self):
        super().__init__(); self.table=nn.Embedding(VOCAB,VOCAB)
    def forward(self,ids): return self.table(ids)

class SingleHeadLM(nn.Module):
    def __init__(self,d=64):
        super().__init__()
        self.token=nn.Embedding(VOCAB,d); self.position=nn.Embedding(CONTEXT,d)
        self.q=nn.Linear(d,d,bias=False); self.k=nn.Linear(d,d,bias=False); self.v=nn.Linear(d,d,bias=False)
        self.head=nn.Linear(d,VOCAB)
    def forward(self,ids,return_attention=False):
        h=self.token(ids)+self.position(torch.arange(ids.size(1),device=ids.device))
        out,a=scaled_attention(self.q(h),self.k(h),self.v(h))
        logits=self.head(out)
        return (logits,a) if return_attention else logits

## §4 · 언어 모델 학습과 생성 함수

평가는 고정 window의 token CE를 집계합니다. 마지막 불완전 window는 제외합니다. 생성은 매번 마지막 CONTEXT 문자를 다시 계산합니다.

In [5]:
@torch.no_grad()
def lm_eval(model,split='val'):
    was_training=model.training; model.eval(); total=count=0
    data=tokens[split]
    # 고정 연속 window: 평가 target을 중복 계산하지 않으며 남는 마지막 불완전 window는 제외.
    for start in range(0,len(data)-CONTEXT,CONTEXT*BATCH):
        starts=range(start,min(start+CONTEXT*BATCH,len(data)-CONTEXT),CONTEXT)
        x=torch.stack([data[i:i+CONTEXT] for i in starts]).to(DEVICE)
        y=torch.stack([data[i+1:i+CONTEXT+1] for i in starts]).to(DEVICE)
        logits=model(x)
        total+=F.cross_entropy(logits.reshape(-1,VOCAB),y.reshape(-1),reduction='sum').item()
        count+=y.numel()
    model.train(was_training)
    return total/count

def fit_lm(factory,steps=STEPS):
    seed_all(); model=factory().to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=3e-3,weight_decay=.01)
    history=[]; best=float('inf'); state=None; start=time.perf_counter()
    for step in range(1,steps+1):
        model.train(); x,y=lm_batch(); opt.zero_grad(set_to_none=True)
        loss=F.cross_entropy(model(x).reshape(-1,VOCAB),y.reshape(-1))
        assert torch.isfinite(loss)
        loss.backward(); norm=nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        if step==1 or step%max(1,steps//5)==0 or step==steps:
            val=lm_eval(model)
            history.append(dict(step=step,train_ce=loss.item(),val_ce=val,grad_norm=float(norm)))
            if val<best: best=val; state=cpu_state(model)
            print(f'step {step}: train={loss.item():.3f} val={val:.3f}')
    model.load_state_dict(state); model.eval()
    return model,history,time.perf_counter()-start

@torch.no_grad()
def generate(model,prompt='the ',new_tokens=120,temperature=1.0,top_k=None,greedy=False,seed=123):
    assert temperature>0 and len(prompt)>0
    seed_all(seed); model.eval()
    ids=torch.tensor([encode(prompt)],device=DEVICE)
    for _ in range(new_tokens):
        logits=model(ids[:,-CONTEXT:])[:,-1,:]/temperature
        if top_k is not None:
            k=max(1,min(int(top_k),VOCAB)); values,index=logits.topk(k)
            logits=torch.full_like(logits,float('-inf')).scatter(1,index,values)
        nxt=logits.argmax(-1,keepdim=True) if greedy else torch.multinomial(logits.softmax(-1),1)
        ids=torch.cat([ids,nxt],dim=1)
    return decode(ids[0])

def assert_causal(model):
    model.eval(); x,_=lm_batch(batch=2); changed=x.clone(); cut=CONTEXT//2
    changed[:,cut:]=(changed[:,cut:]+1)%VOCAB
    with torch.no_grad():
        a=model(x)[:,:cut]; b=model(changed)[:,:cut]
    torch.testing.assert_close(a,b,atol=1e-5,rtol=1e-5)
    print('PASS: changing future tokens does not change earlier logits')

def plot_lm(histories):
    for name,h in histories.items():
        plt.plot([r['step'] for r in h],[r['val_ce'] for r in h],label=name+' val')
        plt.plot([r['step'] for r in h],[r['train_ce'] for r in h],'--',label=name+' train batch')
    plt.xlabel('step'); plt.ylabel('cross entropy'); plt.legend(); plt.grid(alpha=.2); plt.show()

## A · scaled dot-product / 15분

Q,K,V의 마지막 두 축에서 attention을 계산하고 output과 weights를 반환하세요. causal=True이면 미래 key를 차단하세요.

In [6]:
def my_attention(q,k,v,causal=True):
    # TODO
    return None

In [7]:
q,k,v=[torch.randn(2,4,8) for _ in range(3)]
r=my_attention(q,k,v)
if r is None: print('미완성: A를 구현하세요.')
else:
    out,a=r
    torch.testing.assert_close(a.sum(-1),torch.ones(2,4))
    assert a.triu(1).abs().max()==0
    reference=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.)
    torch.testing.assert_close(out,reference,atol=1e-5,rtol=1e-5)
    print('PASS A: reference SDPA match')

미완성: A를 구현하세요.


**기록:** 구현한 식 또는 shape, 검사 결과, 아직 불확실한 점을 적으세요.

_여기에 작성_

## B · next-token window / 10분

1차원 tokens, 시작 i, 길이 t에서 x와 y를 반환하세요. 범위 밖 요청은 AssertionError로 거절하세요.

In [8]:
def next_window(data,i,t):
    # TODO
    return None

In [9]:
r=next_window(torch.arange(10),2,4)
if r is None: print('미완성: B를 구현하세요.')
else:
    x,y=r
    assert x.tolist()==[2,3,4,5] and y.tolist()==[3,4,5,6]
    try: next_window(torch.arange(10),8,3)
    except AssertionError: print('PASS B')
    else: raise AssertionError('out-of-range request must fail')

미완성: B를 구현하세요.


**기록:** 구현한 식 또는 shape, 검사 결과, 아직 불확실한 점을 적으세요.

_여기에 작성_

## C · causal 누출 탐지 / 15분

주어진 모델과 ids에서 cut 이후 토큰을 바꾸고 cut 이전 logits 차이의 최댓값을 반환하세요. eval, no_grad를 사용하세요.

In [10]:
def future_difference(model,ids,cut):
    # TODO
    return None

In [11]:
m=SingleHeadLM().to(DEVICE); x,_=lm_batch(batch=2)
r=future_difference(m,x,CONTEXT//2)
if r is None: print('미완성: C를 구현하세요.')
else:
    assert r<1e-5
    class Leaky(nn.Module):
        def forward(self,x): return F.one_hot(x,VOCAB).float().mean(1,keepdim=True).expand(-1,x.size(1),-1)
    leak=future_difference(Leaky().to(DEVICE),torch.zeros_like(x),CONTEXT//2)
    assert leak>0
    print('PASS C:',r,'leaky model difference:',leak)

미완성: C를 구현하세요.


**기록:** 구현한 식 또는 shape, 검사 결과, 아직 불확실한 점을 적으세요.

_여기에 작성_

## D · 문맥 길이 실험 / 30–45분

같은 SingleHeadLM으로 context=8,32를 비교하세요. 처리 토큰 예산을 맞추기 위해 steps를 반비례 조절하세요. 비교 후 전역 CONTEXT를 원래 값으로 복원하세요. 결과에 context,steps,tokens,val_ce를 기록하세요.

In [12]:
def context_experiment():
    # TODO: 전역 CONTEXT와 lm_batch 기본 인자 주의
    return None

In [13]:
rows=context_experiment()
if rows is None: print('미완성: D는 도전 과제입니다.')
else:
    assert rows[0]['tokens']==rows[1]['tokens']
    assert all(np.isfinite(r['val_ce']) for r in rows)
    display(rows)

미완성: D는 도전 과제입니다.


**기록:** 구현한 식 또는 shape, 검사 결과, 아직 불확실한 점을 적으세요.

_여기에 작성_

## 비교 실험 해석 · 5문장 필수 틀

1. 나의 질문과 바꾼 변수는 …
2. 고정한 조건(분할, seed, 예산, 평가)은 …
3. 실제 관측 결과는 … (숫자·그림 번호를 근거로)
4. 이 결과만으로 말할 수 없는 것은 …
5. 다음에 한 가지만 더 검증한다면 …

| 조건 | seed | 학습 예산 | params | validation 지표 | 시간 | 관찰 |
|---|---|---|---|---|---|---|
| 기준 | | | | | | |
| 변경 | | | | | | |

### 자기 점검 기준 (100점 환산, 성적 부여 목적 아님)
- 구현의 shape·수식·gradient·mask 계약 30
- 통제 조건과 재현 가능한 코드 25
- 실제 결과 표·그림과 실패 사례 25
- 과장 없는 해석과 한계·후속 실험 20

고정 정확도나 생성 품질 기준은 없습니다. 개선되지 않은 실험도 근거를 잘 남기면 좋은 결과물입니다.